In [1]:
!pip install sentence-transformers faiss-cpu pypdf pandas tqdm scikit-learn openai

#### Importing all the required libraries

In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from pypdf import PdfReader
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

from openai import OpenAI

from IPython.display import display, Markdown
import re

#### Loading all the documents, I use my Linear Algebra Notes from course MA110 for the sake of this assessment

In [3]:
documents = []

for i in range(1, 21):

    filename = f"Lecture{i}_D1.pdf"
    reader = PdfReader(filename)

    for page in reader.pages:
        text = page.extract_text()

        if text:
            documents.append(text)

print("Total pages loaded:", len(documents))

Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 125 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 120 0 (offset 0)


Total pages loaded: 330


#### Chunking the dataset into smaller parts so that they can be handled better

In [4]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [5]:
chunks_A = []

for doc in documents:
    chunks_A.extend(chunk_text(doc, 500, 50))

print("Total chunks Strategy A:", len(chunks_A))

chunks_B = []

for doc in documents:
    chunks_B.extend(chunk_text(doc, 1000, 100))

print("Total chunks Strategy B:", len(chunks_B))

Total chunks Strategy A: 642
Total chunks Strategy B: 349


#### Creating embeddings for the chunks generated above

In [6]:
embedding_model = SentenceTransformer("BAAI/bge-base-en")

embeddings_A = embedding_model.encode(chunks_A)
embeddings_B = embedding_model.encode(chunks_B)

In [7]:
dimension = embeddings_A.shape[1]

index_A = faiss.IndexFlatL2(dimension)
index_A.add(embeddings_A.astype("float32"))

index_B = faiss.IndexFlatL2(dimension)
index_B.add(embeddings_B.astype("float32"))

print("Embedding dimension:", dimension)
print("FAISS indices created")

Embedding dimension: 768
FAISS indices created


#### Now, we retrieve the top k most similar chunks to the input query

In [8]:
k = 3

def retrieve(index, chunks, query, k=k):
    query_embedding = embedding_model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, k)
    results = [chunks[i] for i in indices[0]]

    return results

#### Implementing Hybrid search ie both keyword retrieval and semantic vectorial simialrity retrieval to get better context from the posed query

In [9]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunks_A)

def keyword_retrieve(query, k):
    query_vec = vectorizer.transform([query])
    scores = (tfidf_matrix @ query_vec.T).toarray().flatten()
    top_indices = np.argsort(scores)[::-1][:k]

    return [chunks_A[i] for i in top_indices]

def hybrid_retrieve(query, k):
    vector_results = retrieve(index_A, chunks_A, query, k)
    keyword_results = keyword_retrieve(query, k)
    combined = list(dict.fromkeys(vector_results + keyword_results))

    return combined[:k]

#### Now, writing a query rewriter so as to pose a better worded query to the LLM to get a better response

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

def rewrite_query(query):

    prompt = f"""
                Rewrite the following query so that it is clearer for retrieving information
                from linear algebra lecture notes.
                
                Query:
                {query}
              """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

#### Now using the context generated above to pass it to the LLM as context and generate a better answer to the posed query

In [12]:
def generate_answer(query):
    rewritten_query = rewrite_query(query)
    retrieved = hybrid_retrieve(rewritten_query, k)
    context = "\n\n".join(retrieved)

    prompt = f"""
                Use the context below to answer the question.
                
                Context:
                {context}
                
                Question:
                {query}
                
                Use LaTeX math notation.
                Use $...$ for inline math and $$...$$ for equations.
              """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content
    return answer

def rag_answer(query):
    answer = generate_answer(query)
    answer = re.sub(r"\\\((.*?)\\\)", r"$\1$", answer)
    answer = re.sub(r"\\\[(.*?)\\\]", r"$$\1$$", answer)
    display(Markdown(answer))

rag_answer("Explain eigenvalues.")

Eigenvalues are fundamental concepts in linear algebra that provide insights into the properties of linear transformations represented by matrices. Given a square matrix $ A $, an eigenvalue $ \lambda $ is a scalar such that there exists a non-zero vector $ x $ (called an eigenvector) satisfying the equation:

$$
A x = \lambda x
$$

In this equation:

- $ A $ is the matrix acting on the vector $ x $.
- $ \lambda $ is the eigenvalue associated with the eigenvector $ x $.
- $ x $ is an eigenvector corresponding to the eigenvalue $ \lambda $.

The significance of eigenvalues lies in their ability to characterize the behavior of the matrix $ A $ under various transformations. For example, when the matrix $ A $ is applied to the eigenvector $ x $, the output is simply a scaled version of $ x $ (scaled by $ \lambda $), indicating that eigenvectors point in directions that remain unchanged (up to scaling) under the transformation defined by $ A $.

To find the eigenvalues of $ A $, we can rearrange the eigenvalue equation:

$$
A x - \lambda x = 0
$$

This can be rewritten as:

$$
(A - \lambda I) x = 0
$$

where $ I $ is the identity matrix of the same dimension as $ A $. The above equation has non-trivial solutions (non-zero $ x $) if and only if the determinant is zero:

$$
\text{det}(A - \lambda I) = 0
$$

The solutions $ \lambda $ obtained from this equation are the eigenvalues of $ A $. Each eigenvalue can have one or more corresponding eigenvectors, forming a fundamental basis for the vector space. 

In summary, eigenvalues provide crucial information about the scaling factors in linear transformations, and they are pivotal in various applications across mathematics, physics, and engineering.

#### Comparing the performance of strategies A and B on a sample dataset

In [13]:
evaluation_data = [
    {
        "query": "What is a vector space?",
        "expected_keywords": ["vector space", "scalar multiplication", "addition"]
    },
    {
        "query": "Explain eigenvalues.",
        "expected_keywords": ["eigenvalue", "eigenvector", "lambda"]
    },
    {
        "query": "What is matrix rank?",
        "expected_keywords": ["rank", "linearly independent"]
    }
]

def evaluate_retrieval(index, chunks, k):
    rows = []

    for item in evaluation_data:
        query = item["query"]
        keywords = item["expected_keywords"]

        retrieved = retrieve(index, chunks, query, k)
        relevant_count = 0

        for chunk in retrieved:
            text = chunk.lower()
            if any(keyword.lower() in text for keyword in keywords):
                relevant_count += 1

        precision = relevant_count / k
        recall = 1 if relevant_count > 0 else 0

        rows.append({
            "query": query,
            "relevant_chunks": relevant_count,
            f"precision@{k}": precision,
            "recall": recall
        })

    return pd.DataFrame(rows)

results_A = evaluate_retrieval(index_A, chunks_A, k)
results_B = evaluate_retrieval(index_B, chunks_B, k)

print("Strategy A Results")
display(results_A)

print("Strategy B Results")
display(results_B)

Strategy A Results


,query,relevant_chunks,precision@3,recall
0,What is a vector space?,1,0.333333,1
1,Explain eigenvalues.,3,1.000000,1
2,What is matrix rank?,3,1.000000,1


Strategy B Results


,query,relevant_chunks,precision@3,recall
0,What is a vector space?,1,0.333333,1
1,Explain eigenvalues.,3,1.000000,1
2,What is matrix rank?,3,1.000000,1


#### Checking for hallucination in the results by checking how similar the generated answer is to the original context using cosine similarity

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

def detect_hallucination(query):
    retrieved = hybrid_retrieve(query, k) # retrieve context
    
    context = " ".join(retrieved)
    answer = generate_answer(query) # generate model answer

    context_embedding = embedding_model.encode([context]) # generate embeddings
    answer_embedding = embedding_model.encode([answer])

    similarity = cosine_similarity(answer_embedding, context_embedding)[0][0] # compute semantic similarity
    hallucinated = similarity < 0.5 # threshold (can be tuned)

    return {
        "query": query,
        "similarity_score": float(similarity),
        "hallucination_detected": hallucinated
    }

hallucination_results = []
for item in evaluation_data:
    result = detect_hallucination(item["query"])
    hallucination_results.append(result)

print(pd.DataFrame(hallucination_results))

                     query  similarity_score  hallucination_detected
0  What is a vector space?          0.862344                   False
1     Explain eigenvalues.          0.903409                   False
2     What is matrix rank?          0.945583                   False
